# Lecture 2. Interpreter, dependencies and the lockfile

Everything taught tonight, in the order it was taught. Run it top to bottom.

The Unicorn Adoption Bureau is imported from the package rather than copied in,
so there is exactly one version of it. Short teaching fragments are written out
in full, because you should be able to read them without opening another file.

In [ ]:
# beat: 1630 which-python
import sys

print("interpreter:", sys.executable)
print("version:    ", sys.version.split()[0])
print("first three places import looks:")
for entry in sys.path[:3]:
    print("   ", entry or "(the current folder)")

In [ ]:
# beat: 1643 stdlib-against-pypi
import json

print("json ships with Python:", json.__file__)

try:
    import pandas
except ModuleNotFoundError as missing:
    print("pandas does not:", missing)

In [ ]:
# beat: 1647 where-did-it-go
from pathlib import Path

venv = Path(sys.prefix)
print("this environment lives at:", venv)
site_packages = next(venv.glob("lib/python*/site-packages"), None)
print("packages land in:         ", site_packages)
print("and that folder is on sys.path:", str(site_packages) in sys.path)

In [ ]:
# beat: 1749 generated-decide
from unicorn_adoption_bureau.generated import decide_as_generated as generated
import inspect

source = inspect.getsource(generated.decide)
print(f"the assistant wrote {len(source.splitlines())} lines")
print("\n".join(source.splitlines()[:14]))

In [ ]:
# beat: 1806 line-count
from unicorn_adoption_bureau import rules

after = inspect.getsource(rules.decide)
print("generated:  ", len(source.splitlines()), "lines")
print("after the three buckets:", len(after.splitlines()), "lines")

In [ ]:
# beat: 1811 fail-fast
from unicorn_adoption_bureau.rules import Application, MissingAnswer, decide

unanswered = Application(
    garden_m2=80.0, glitter_tolerance=None, hours_at_home=30, floor=1, has_lift=False
)
try:
    decide(unanswered)
except MissingAnswer as stopped:
    print("stopped, rather than scoring it as perfect:", stopped)

## Exercise 1. Make the failing check pass

One character in the function below is wrong. The Bureau's rule is that a
garden of **at least** fifty square metres is enough, and the third assertion
says so. Run the cell, read the failure, then fix it.

In [ ]:
MIN_GARDEN_M2 = 50.0


def garden_is_big_enough(garden_m2: float) -> bool:
    return garden_m2 > MIN_GARDEN_M2  # TODO one character is wrong


assert garden_is_big_enough(50.1)
assert not garden_is_big_enough(49.9)
assert garden_is_big_enough(50.0), "a garden of exactly the minimum is big enough"
print("all three pass")

<details>

<summary>Show the solution</summary>

```python
def garden_is_big_enough(garden_m2: float) -> bool:
    """At least the minimum means at least, so the comparison is inclusive."""
    return garden_m2 >= MIN_GARDEN_M2
```

The applicant with exactly the required garden is the one the rule was written
for, and the one a strict comparison silently refuses. Boundaries are where
rules are wrong, which is why the tests you write tonight go there first.

</details>

## Exercise 2. Write the missing test

The Bureau refuses a unicorn above the second floor when there is no lift,
whatever the rest of the application says. No test covers that. Write one.

You have `Application`, `decide` and `pytest` already imported above.

In [ ]:
def test_the_floor_rule_overrides_a_perfect_score():
    ...  # TODO


test_the_floor_rule_overrides_a_perfect_score()
print("it passes")

<details>

<summary>Show the solution</summary>

```python
def test_the_floor_rule_overrides_a_perfect_score(application):
    changed = Application(**{**vars(application), "floor": 4, "has_lift": False})
    assert not decide(changed).approved
```

A policy rule is one the score cannot outvote, so the test has to start from an
application that would otherwise be approved. A test that starts from a bad
application would pass whether the rule existed or not.

</details>